## Loading and Combining Datasets

In [10]:
import polars as pl
import pandas as pd
import polars.selectors as cs

In [11]:
all_df_filepath = r"..\data\temzy\all data .xlsx"

In [12]:
def load_and_comb_dfs(filepath: str)-> pl.DataFrame:
    try:
        comb_df = pl.read_excel(
            filepath,
            sheet_id=0,
            infer_schema_length=None,
        )

        all_dfs_list = []

        for name, sheet in comb_df.items():
            sheet = sheet.with_columns(pl.lit(name).alias('country'))
            all_dfs_list.append(sheet)
            
        df = pl.concat(all_dfs_list, how='diagonal')
        df = df.rename({'__UNNAMED__0': 'year'}).drop(df.columns[-1])
        return df
    except FileNotFoundError:
        print("Data file not found in directory! Kindly ensure file exists in folder path.")
    except Exception as e:
        print(f"Error encountered: {e}")

final_df = load_and_comb_dfs(filepath=all_df_filepath)
final_df.describe()
final_df

year,Real GDP (USD million) (% growth),Median Disposable Income per Household (USD),Unemployment Rate (% of economically active population),Population Living Below International Poverty Line ($1.90 a Day),Inflation,Imports (USD million),Exports (USD million),country
i64,f64,f64,f64,f64,f64,f64,f64,str
1993,484743.6,4189.9,6.523,10.5,1956.415,27604.4,38554.8,"""Brazil"""
1994,604102.9,5121.8,6.72,9.4,2188.422,36192.3,43545.1,"""Brazil"""
1995,778788.9,7651.9,7.24,8.4,71.039,54137.4,46506.3,"""Brazil"""
1996,850426.4,8312.4,8.199,7.9,15.757,56981.0,47746.7,"""Brazil"""
1997,883207.8,8428.1,9.188,7.7,6.925,64242.2,52994.3,"""Brazil"""
…,…,…,…,…,…,…,…,…
2018,245173.1,4797.6,2.002,2.5,3.539,236868.9,243698.7,"""Vietnam"""
2019,261915.0,5124.9,2.163,2.2,2.795,254092.8,264340.8,"""Vietnam"""
2020,271193.3,5225.5,2.455,2.0,3.22,262673.3,282528.8,"""Vietnam"""


In [13]:
clean_df = final_df.rename({
    'Real GDP (USD million) (% growth)': 'gdp_growth',
    'Median Disposable Income per Household (USD)': 'household_income',
    'Unemployment Rate (% of economically active population)': 'unemployment_rate',
    'Population Living Below International Poverty Line ($1.90 a Day)': 'poverty_rate',
    'Inflation': 'inflation',
    'Imports (USD million)': 'imports',
    'Exports (USD million)': 'exports'
})

# clean_df.sort(['country', 'year'])


In [14]:
clean_df

year,gdp_growth,household_income,unemployment_rate,poverty_rate,inflation,imports,exports,country
i64,f64,f64,f64,f64,f64,f64,f64,str
1993,484743.6,4189.9,6.523,10.5,1956.415,27604.4,38554.8,"""Brazil"""
1994,604102.9,5121.8,6.72,9.4,2188.422,36192.3,43545.1,"""Brazil"""
1995,778788.9,7651.9,7.24,8.4,71.039,54137.4,46506.3,"""Brazil"""
1996,850426.4,8312.4,8.199,7.9,15.757,56981.0,47746.7,"""Brazil"""
1997,883207.8,8428.1,9.188,7.7,6.925,64242.2,52994.3,"""Brazil"""
…,…,…,…,…,…,…,…,…
2018,245173.1,4797.6,2.002,2.5,3.539,236868.9,243698.7,"""Vietnam"""
2019,261915.0,5124.9,2.163,2.2,2.795,254092.8,264340.8,"""Vietnam"""
2020,271193.3,5225.5,2.455,2.0,3.22,262673.3,282528.8,"""Vietnam"""


possible feature engineering
- export - import = trade_balance

### Log Dependent Var (Gdp Growth)

In [15]:
# clean_df = (clean_df
# .sort(['country', 'year'])
# .with_columns([
#    pl.col("gdp_growth").log().diff().over("country").alias('gdp_growth_log')
# ])
# )
# clean_df = clean_df.fill_null(0)

### Separate Collated DFs by Country

In [16]:
countries = clean_df.get_column("country").unique().to_list()

country_data = {
    country: clean_df.filter(pl.col("country") == country).sort("year")
    for country in countries
}

country_data

{'India': shape: (30, 9)
 ┌──────┬────────────┬───────────────┬──────────────┬───┬───────────┬──────────┬──────────┬─────────┐
 │ year ┆ gdp_growth ┆ household_inc ┆ unemployment ┆ … ┆ inflation ┆ imports  ┆ exports  ┆ country │
 │ ---  ┆ ---        ┆ ome           ┆ _rate        ┆   ┆ ---       ┆ ---      ┆ ---      ┆ ---     │
 │ i64  ┆ f64        ┆ ---           ┆ ---          ┆   ┆ f64       ┆ f64      ┆ f64      ┆ str     │
 │      ┆            ┆ f64           ┆ f64          ┆   ┆           ┆          ┆          ┆         │
 ╞══════╪════════════╪═══════════════╪══════════════╪═══╪═══════════╪══════════╪══════════╪═════════╡
 │ 1993 ┆ 287273.8   ┆ 1120.4        ┆ 5.612        ┆ … ┆ 6.362     ┆ 22788.4  ┆ 21571.6  ┆ India   │
 │ 1994 ┆ 327525.5   ┆ 1277.2        ┆ 5.631        ┆ … ┆ 10.212    ┆ 26842.7  ┆ 25021.8  ┆ India   │
 │ 1995 ┆ 371782.7   ┆ 1396.7        ┆ 5.636        ┆ … ┆ 10.225    ┆ 34706.9  ┆ 30630.0  ┆ India   │
 │ 1996 ┆ 393646.9   ┆ 1472.3        ┆ 5.652        ┆ … ┆

# Diagnostics and Validation

In [17]:
import polars as pl
import numpy as np
import pandas as pd
import statsmodels.api as sm

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.anova import anova_lm

## Stationarity
- Logarithmic differencing is applied to all float variables to convert them into growth rates, ensuring stationarity, reducing trends, and maintaining consistency across the dataset.
 
- It was previously applied only to specific columns in the previous version.
  - gdp_growth, household_income, exports and imports
(14-04-2026)

Further actions: (27-04-2026) 
  + Log only normal continous data, leave rates (unemployment, poverty and inflation) unlogged.
  + you can apply differencing only if possible.

**Date Last Modified: 27-04-2026**

### Functions

In [18]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss

def clean_series(series):
    s = pd.Series(series)
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    return s

def run_adf(series, name="variable"):
    series = clean_series(series)

    if len(series) < 3:
        return {
            "variable": name,
            "test": "ADF",
            "statistic": np.nan,
            "p_value": np.nan,
            "lags_used": np.nan,
            "n_obs": len(series),
            "note": "Too few observations"
        }

    if series.nunique() <= 1 or np.isclose(series.std(ddof=0), 0, equal_nan=True):
        return {
            "variable": name,
            "test": "ADF",
            "statistic": np.nan,
            "p_value": np.nan,
            "lags_used": np.nan,
            "n_obs": len(series),
            "note": "Constant / invalid series"
        }

    try:
        result = adfuller(series)
        return {
            "variable": name,
            "test": "ADF",
            "statistic": result[0],
            "p_value": result[1],
            "lags_used": result[2],
            "n_obs": result[3],
            "note": ""
        }
    except Exception as e:
        return {
            "variable": name,
            "test": "ADF",
            "statistic": np.nan,
            "p_value": np.nan,
            "lags_used": np.nan,
            "n_obs": len(series),
            "note": str(e)
        }

def run_kpss(series, name="variable"):
    series = clean_series(series)

    if len(series) < 3:
        return {
            "variable": name,
            "test": "KPSS",
            "statistic": np.nan,
            "p_value": np.nan,
            "lags_used": np.nan,
            "note": "Too few observations"
        }

    if series.nunique() <= 1 or np.isclose(series.std(ddof=0), 0, equal_nan=True):
        return {
            "variable": name,
            "test": "KPSS",
            "statistic": np.nan,
            "p_value": np.nan,
            "lags_used": np.nan,
            "note": "Constant / invalid series"
        }

    try:
        result = kpss(series, regression="c", nlags="auto")
        return {
            "variable": name,
            "test": "KPSS",
            "statistic": result[0],
            "p_value": result[1],
            "lags_used": result[2],
            "note": ""
        }
    except Exception as e:
        return {
            "variable": name,
            "test": "KPSS",
            "statistic": np.nan,
            "p_value": np.nan,
            "lags_used": np.nan,
            "note": str(e)
        }

def stationarity_check(variables: list, data_dict) -> pl.dataframe:
    stationarity_results = []

    for country, data in data_dict.items():
        for col in variables:
            values = data.get_column(col).to_list()

            adf_result = run_adf(values, col)
            adf_result['statistic'] = round(adf_result['statistic'],3)
            adf_result['p_value'] = round(adf_result['p_value'],3)
            adf_result["country"] = country
            stationarity_results.append(adf_result)

            kpss_result = run_kpss(values, col)
            kpss_result['statistic'] = round(kpss_result['statistic'], 3)
            kpss_result['p_value'] = round(kpss_result['p_value'], 3)
            kpss_result["country"] = country
            stationarity_results.append(kpss_result)

    stationarity_df = pl.DataFrame(stationarity_results)
    return stationarity_df

def statinarity_data_transformation(data_dict: dict) -> dict:
    trans_data_dict = {}

    try:
        for country, data in data_dict.items():
            
            df = (
                data
                .sort("year")
                .with_columns(
                    cs.float().cast(pl.Float64).log().diff()
            )
                .drop_nulls(
                    cs.float()
                )
            )
            
            trans_data_dict[country] = df

        return trans_data_dict
    except Exception as e:
        print(f"Error encountered: {e}")


### Raw DF Stationarity Check

In [19]:
raw_df_variables = ["gdp_growth", "household_income", "unemployment_rate", "poverty_rate", "inflation", "imports", "exports"]

raw_df_stationarity_check = stationarity_check(variables=raw_df_variables, data_dict=country_data)

C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_17316\3161929974.py:81: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result = kpss(series, regression="c", nlags="auto")
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_17316\3161929974.py:81: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result = kpss(series, regression="c", nlags="auto")
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_17316\3161929974.py:81: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(series, regression="c", nlags="auto")
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_17316\3161929974.py:81: InterpolationWarning: The test statistic is outside of th

In [20]:
ad = raw_df_stationarity_check
ac = ad.to_pandas()
ac.to_csv(r"..\data\ecoms_results\raw_data_stationarity_check.csv", index=False)

In [21]:
stationarity_pd = ad.to_pandas()

decision_map = {}

for country in countries:
    decision_map[country] = {}
    for var in raw_df_variables:
        adf_p = stationarity_pd[
            (stationarity_pd["country"] == country) &
            (stationarity_pd["variable"] == var) &
            (stationarity_pd["test"] == "ADF")
        ]["p_value"].iloc[0]

        kpss_p = stationarity_pd[
            (stationarity_pd["country"] == country) &
            (stationarity_pd["variable"] == var) &
            (stationarity_pd["test"] == "KPSS")
        ]["p_value"].iloc[0]

        decision_map[country][var] = {
            "difference": (adf_p > 0.05) or (kpss_p < 0.05),
            "adf_p": adf_p,
            "kpss_p": kpss_p
        }

### Transformation
- logarithmic differencing was applied to all float columns

Observation
- Applying logarithmic differencing to all columns affected variables like poverty with -inf and zero values when calculating kpss stationarity especially for country like Czech Republic with 0.0 poverty rates.
- Modified the ADF and KPSS functions to account for this anomaly. 

Recommendation
- Ask Temzy for further actions.

In [22]:
transformed_data = statinarity_data_transformation(data_dict=country_data)
transformed_data['Canada'].columns

['year',
 'gdp_growth',
 'household_income',
 'unemployment_rate',
 'poverty_rate',
 'inflation',
 'imports',
 'exports',
 'country']

#### Identifying country with Nan after transformation

In [23]:
variables = ["gdp_growth", "household_income", "unemployment_rate", "poverty_rate", "inflation", "imports", "exports"]

for country, df in transformed_data.items():
    print(f"\n--- {country} ---")
    for col in variables:
        s = df.get_column(col).to_pandas().dropna()
        print(col, "len=", len(s), "nunique=", s.nunique(), "std=", round(s.std(ddof=0), 4))


--- India ---
gdp_growth len= 29 nunique= 29 std= 0.0695
household_income len= 29 nunique= 29 std= 0.0662
unemployment_rate len= 29 nunique= 28 std= 0.0403
poverty_rate len= 29 nunique= 29 std= 0.0352
inflation len= 29 nunique= 29 std= 0.3623
imports len= 29 nunique= 29 std= 0.1763
exports len= 29 nunique= 29 std= 0.1407

--- Brazil ---
gdp_growth len= 29 nunique= 29 std= 0.1679
household_income len= 29 nunique= 29 std= 0.1742
unemployment_rate len= 29 nunique= 29 std= 0.1203
poverty_rate len= 29 nunique= 24 std= 0.0473
inflation len= 29 nunique= 29 std= 0.7919
imports len= 29 nunique= 29 std= 0.194
exports len= 29 nunique= 29 std= 0.1358

--- Vietnam ---
gdp_growth len= 29 nunique= 29 std= 0.0632
household_income len= 29 nunique= 29 std= 0.071
unemployment_rate len= 29 nunique= 29 std= 0.2005
poverty_rate len= 29 nunique= 29 std= 0.0505
inflation len= 26 nunique= 26 std= 0.6902
imports len= 29 nunique= 29 std= 0.1202
exports len= 29 nunique= 29 std= 0.0938

--- Canada ---
gdp_growth 

c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [24]:
import numpy as np
import pandas as pd

s = transformed_data["Czech Republic"].get_column("poverty_rate").to_pandas()

print(s)
print("has_nan:", s.isna().any())
print("has_pos_inf:", np.isposinf(s).any())
print("has_neg_inf:", np.isneginf(s).any())
print("finite_count:", np.isfinite(s).sum())
print("non_finite_rows:")
print(s[~np.isfinite(s)])

0    -0.223144
1     0.000000
2    -0.287682
3     0.000000
4     0.000000
5    -0.405465
6     0.000000
7     0.000000
8     0.405465
9     0.000000
10    0.000000
11   -0.405465
12    0.000000
13    0.000000
14   -0.693147
15    0.000000
16    0.000000
17    0.000000
18    0.000000
19    0.000000
20    0.000000
21    0.000000
22        -inf
23         NaN
24         NaN
25         NaN
26         NaN
27         NaN
28         NaN
Name: poverty_rate, dtype: float64
has_nan: True
has_pos_inf: False
has_neg_inf: True
finite_count: 22
non_finite_rows:
22   -inf
23    NaN
24    NaN
25    NaN
26    NaN
27    NaN
28    NaN
Name: poverty_rate, dtype: float64


In [25]:
# transformed_data['Czech Republic']

#### Transformed DF Stationarity Check

In [26]:
variables = [
    "gdp_growth", "household_income", "unemployment_rate",
    "poverty_rate", "inflation", "imports", "exports",
]

trans_df_stationarity_res = stationarity_check(variables=variables, data_dict=transformed_data).to_pandas()
trans_df_stationarity_res.to_csv(r"..\data\ecoms_results\transformed_data_stationarity_check.csv", index=False)

C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_17316\3161929974.py:81: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(series, regression="c", nlags="auto")
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_17316\3161929974.py:81: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(series, regression="c", nlags="auto")
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_17316\3161929974.py:81: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(series, regression="c", nlags="auto")
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_17316\3161929974.py:81: InterpolationWarning: The test statistic is outside of th

## ARDL Check
Goal:
1. ADF & KPSS for stationarity ``Done``
2. ARDL to harmonise stationary and non stationary data  ``Current``
3. Heteroskedasticity (Post-estimation Diagnostic)
4. Multicollinearity (Structural Consideration)

The focus is on ADRL right now.

Observations:
- ic: 'bic' was used in order to ensure model was not overfitting.
- review the effect of unemployment rate on the model.
- Review bounds test effect on the model.
   + We need to know:

        + Cointegration (Bounds Test)
        + F > Upper bound → Cointegration
        + F < Lower bound → No cointegration
        + Between → Inconclusive

**27-04-2026**
- Performed ADRL tests and reviewed with Temzy. 
- ADRL model summary reports to be shared for review. 

**06-05-2026**
- Created and parsed the model summaries into word document for review and possible adjustment.

**Date Modified 06-05-2026**

##### Functions

In [35]:
import pandas as pd
import numpy as np
import polars as pl
from statsmodels.tsa.ardl import ardl_select_order
from statsmodels.tsa.ardl import UECM


def adrl_data_selection(data_dict: dict):
    ardl_raw_data = {}
    ardl_log_data = {}

    for country, df in data_dict.items():
        df = df.sort("year")

        # Raw-level ARDL dataset
        ardl_raw_data[country] = df.select([
            "year",
            "gdp_growth",
            "household_income",
            "unemployment_rate",
            "poverty_rate",
            "inflation",
            "imports",
            "exports"
        ]).drop_nulls()

        # Logged-level ARDL dataset
        df_log = (
            df
            .with_columns([
                pl.col("gdp_growth").cast(pl.Float64).log().alias("log_gdp"),
                pl.col("household_income").cast(pl.Float64).log().alias("log_income"),
                pl.col("imports").cast(pl.Float64).log().alias("log_imports"),
                pl.col("exports").cast(pl.Float64).log().alias("log_exports"),
                (pl.col("imports") + pl.col("exports")).cast(pl.Float64).log().alias("log_trade"),
            ])
            .select([
                "year",
                "log_gdp",
                "log_income",
                "unemployment_rate",
                "poverty_rate",
                "inflation",
                "log_imports",
                "log_exports",
                "log_trade"
            ])
            .drop_nulls()
        )

        ardl_log_data[country] = df_log

    return ardl_raw_data, ardl_log_data


def run_ardl_models(data_dict: dict, y_col, x_cols, maxlag=2, maxorder=2, ic="bic"):
    selected_models = {}
    fitted_models = {}
    model_summaries = {}

    for country, data in data_dict.items():
        pdf = data.to_pandas().dropna()

        y = pdf[y_col]
        X = pdf[x_cols]

        sel = ardl_select_order(
            endog=y,
            maxlag=maxlag,
            exog=X,
            maxorder=maxorder,
            ic=ic,
            trend="c"
        )

        model = sel.model.fit()

        selected_models[country] = sel
        fitted_models[country] = model
        model_summaries[country] = model.summary()

        # print(f"\n--- {country} ---")
        # print("Selected ARDL order:", sel.model.ardl_order)
        # print(model.summary())

    return selected_models, fitted_models, model_summaries


def raw_df_adrl(data_dict: dict):
    raw_y = "gdp_growth"

    raw_predictors = [
        "household_income",
        "unemployment_rate",
        "poverty_rate",
        "inflation",
        "imports",
        "exports"
    ]

    raw_selected, raw_models, raw_model_summaries = run_ardl_models(
        data_dict=data_dict,
        y_col=raw_y,
        x_cols=raw_predictors,
        maxlag=2,
        maxorder=2,
        ic="bic"
    )
    return raw_selected, raw_models, raw_model_summaries


def logged_df_adrl_opt_a(data_dict: dict):
    log_y = "log_gdp"

    log_predictors_full = [
        "log_income",
        "unemployment_rate",
        "poverty_rate",
        "inflation",
        "log_imports",
        "log_exports"
    ]

    log_selected_full, log_models_full, opt_a_model_summaries = run_ardl_models(
        data_dict=data_dict,
        y_col=log_y,
        x_cols=log_predictors_full,
        maxlag=2,
        maxorder=2,
        ic="bic"
    )
    return log_selected_full, log_models_full, opt_a_model_summaries


def logged_df_adrl_opt_b(data_dict: dict):
    log_y = "log_gdp"

    log_predictors_clean = [
        "log_income",
        "unemployment_rate",
        "poverty_rate",
        "inflation",
        "log_trade"
    ]

    log_selected_clean, log_models_clean, opt_b_model_summaries = run_ardl_models(
        data_dict=data_dict,
        y_col="log_gdp",
        x_cols=log_predictors_clean,
        maxlag=2,
        maxorder=2,
        ic="bic"
    )
    return log_selected_clean, log_models_clean, opt_b_model_summaries


def run_bounds_tests(selected_models, case=3):
    bounds_results = {}
    uecm_models = {}

    for country, sel in selected_models.items():
        try:
            uecm_model = UECM.from_ardl(sel.model)
            uecm_res = uecm_model.fit()

            bt = uecm_res.bounds_test(case=case)

            uecm_models[country] = uecm_res
            bounds_results[country] = bt

            print(f"\n--- {country} Bounds Test ---")
            print(bt)

        except Exception as e:
            bounds_results[country] = str(e)
            print(f"\n--- {country} Bounds Test Failed ---")
            print(e)

    return bounds_results, uecm_models

##### ADRL Runs

In [36]:
raw_df, logged_df = adrl_data_selection(data_dict = country_data)

## Raw df ADRL
raw_df_selected, raw_df_models, raw_model_summaries = raw_df_adrl(raw_df)


In [37]:
## Logged data option A ADRL
log_df_opt_a_selected, log_df_opt_a_models, opt_a_model_summaries = logged_df_adrl_opt_a(logged_df)

In [38]:
## Logged data option B ADRL
log_df_opt_b_selected, log_df_opt_b_models, opt_b_model_summaries = logged_df_adrl_opt_b(logged_df)

### Parsing Model Summaries to Word Document

In [43]:
from docx import Document
from docx.shared import Pt

def convert_summary_to_word_document(model_summary_dict: dict, document_name: str):

    doc = Document()
    for country, summary_obj in model_summary_dict.items():
        summary_string = summary_obj.as_text() 
        lines = summary_string.split('\n')
        
        doc.add_heading(f'Model Report: {country}', level=1)
        p = doc.add_paragraph()
        run = p.add_run(summary_string)
        
        run.font.name = 'Times New Roman'
        run.font.size = Pt(8)
        doc.add_page_break()

    doc.save(f"../data/temzy/{document_name}.docx")


In [44]:
convert_summary_to_word_document(raw_model_summaries, document_name="raw_model_adrl_report")
convert_summary_to_word_document(opt_a_model_summaries, document_name="log_option_a_adrl_report")
convert_summary_to_word_document(opt_b_model_summaries, document_name="log_option_b_adrl_report")

##### Bounds test using UECM

In [60]:
## raw df adrl selected model bounds test. 

run_bounds_tests(raw_df_selected)


--- Mexico Bounds Test ---
BoundsTestResult
Stat: 1.72789
Upper P-value: 0.643
Lower P-value: 0.343
Null: No Cointegration
Alternative: Possible Cointegration


--- India Bounds Test ---
BoundsTestResult
Stat: 21.77472
Upper P-value: 0
Lower P-value: 0
Null: No Cointegration
Alternative: Possible Cointegration


--- Colombia Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Canada Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Egypt Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- South Africa Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Germany Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Vietnam Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Brazil Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Czech Republic Bounds Test Failed -

c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: imports, inflation, poverty_rate, unemployment_rate.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: imports, inflation, poverty_rate, unemployment_rate.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: inflation.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive

({'Mexico': BoundsTestResult
  Stat: 1.72789
  Upper P-value: 0.643
  Lower P-value: 0.343
  Null: No Cointegration
  Alternative: Possible Cointegration,
  'India': BoundsTestResult
  Stat: 21.77472
  Upper P-value: 0
  Lower P-value: 0
  Null: No Cointegration
  Alternative: Possible Cointegration,
  'Colombia': 'All included exog variables must have a lag length >= 1',
  'Canada': 'All included exog variables must have a lag length >= 1',
  'Egypt': 'All included exog variables must have a lag length >= 1',
  'South Africa': 'All included exog variables must have a lag length >= 1',
  'Germany': 'All included exog variables must have a lag length >= 1',
  'Vietnam': 'All included exog variables must have a lag length >= 1',
  'Brazil': 'All included exog variables must have a lag length >= 1',
  'Czech Republic': 'All included exog variables must have a lag length >= 1'},
 {'Mexico': <statsmodels.tsa.ardl.model.UECMResultsWrapper at 0x230d44c5b20>,
  'India': <statsmodels.tsa.ardl.m

In [68]:
run_bounds_tests(log_df_opt_a_selected)

c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: log_imports, unemployment_rate.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: log_exports, log_imports.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: log_imports, unemployment_rate.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive\Documents\DS\research_proj


--- Mexico Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- India Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Colombia Bounds Test Failed ---
All included exog variables must have a lag length >= 1


c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: inflation, log_imports.
  return _format_order(self.data.orig_exog, order, self._causal)



--- Canada Bounds Test ---
BoundsTestResult
Stat: 3.34637
Upper P-value: 0.101
Lower P-value: 0.0117
Null: No Cointegration
Alternative: Possible Cointegration


--- Egypt Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- South Africa Bounds Test ---
BoundsTestResult
Stat: 8.42556
Upper P-value: 1.8e-07
Lower P-value: 8.8e-10
Null: No Cointegration
Alternative: Possible Cointegration


--- Germany Bounds Test ---
BoundsTestResult
Stat: 10.42648
Upper P-value: 6.61e-10
Lower P-value: 2.43e-12
Null: No Cointegration
Alternative: Possible Cointegration


--- Vietnam Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Brazil Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Czech Republic Bounds Test Failed ---
All included exog variables must have a lag length >= 1


c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: log_exports, unemployment_rate.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: unemployment_rate, inflation, log_imports.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: unemployment_rate, inflation, poverty_rate, log_imports.
  return _format_order(self.data.orig_exog, order, self._causal)


({'Mexico': 'All included exog variables must have a lag length >= 1',
  'India': 'All included exog variables must have a lag length >= 1',
  'Colombia': 'All included exog variables must have a lag length >= 1',
  'Canada': BoundsTestResult
  Stat: 3.34637
  Upper P-value: 0.101
  Lower P-value: 0.0117
  Null: No Cointegration
  Alternative: Possible Cointegration,
  'Egypt': 'All included exog variables must have a lag length >= 1',
  'South Africa': BoundsTestResult
  Stat: 8.42556
  Upper P-value: 1.8e-07
  Lower P-value: 8.8e-10
  Null: No Cointegration
  Alternative: Possible Cointegration,
  'Germany': BoundsTestResult
  Stat: 10.42648
  Upper P-value: 6.61e-10
  Lower P-value: 2.43e-12
  Null: No Cointegration
  Alternative: Possible Cointegration,
  'Vietnam': 'All included exog variables must have a lag length >= 1',
  'Brazil': 'All included exog variables must have a lag length >= 1',
  'Czech Republic': 'All included exog variables must have a lag length >= 1'},
 {'Canada

In [69]:
run_bounds_tests(log_df_opt_b_selected)

c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: unemployment_rate.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: log_trade.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: unemployment_rate.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\


--- Mexico Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- India Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Colombia Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Canada Bounds Test ---
BoundsTestResult
Stat: 3.25348
Upper P-value: 0.116
Lower P-value: 0.0142
Null: No Cointegration
Alternative: Possible Cointegration


--- Egypt Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- South Africa Bounds Test ---
BoundsTestResult
Stat: 10.73038
Upper P-value: 4.58e-06
Lower P-value: 3.9e-07
Null: No Cointegration
Alternative: Possible Cointegration


--- Germany Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Vietnam Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Brazil Bounds Test Failed ---
All included exog variables must have a lag length >= 1

--- Czech Republic Bounds

c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: log_trade, inflation, poverty_rate.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: unemployment_rate.
  return _format_order(self.data.orig_exog, order, self._causal)
c:\Users\APIN PC\OneDrive\Documents\DS\research_projects\.venv\Lib\site-packages\statsmodels\tsa\ardl\model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: inflation, poverty_rate, unemployment_rate.
  return _format_order(self.data.orig_exog, order, self._causal)


({'Mexico': 'All included exog variables must have a lag length >= 1',
  'India': 'All included exog variables must have a lag length >= 1',
  'Colombia': 'All included exog variables must have a lag length >= 1',
  'Canada': BoundsTestResult
  Stat: 3.25348
  Upper P-value: 0.116
  Lower P-value: 0.0142
  Null: No Cointegration
  Alternative: Possible Cointegration,
  'Egypt': 'All included exog variables must have a lag length >= 1',
  'South Africa': BoundsTestResult
  Stat: 10.73038
  Upper P-value: 4.58e-06
  Lower P-value: 3.9e-07
  Null: No Cointegration
  Alternative: Possible Cointegration,
  'Germany': 'All included exog variables must have a lag length >= 1',
  'Vietnam': 'All included exog variables must have a lag length >= 1',
  'Brazil': 'All included exog variables must have a lag length >= 1',
  'Czech Republic': 'All included exog variables must have a lag length >= 1'},
 {'Canada': <statsmodels.tsa.ardl.model.UECMResultsWrapper at 0x230d74c0d10>,
  'South Africa': <s

### Standardization

In [58]:
standardized_data = {}

predictors = [
    "income_growth",
    "imports_growth",
    "exports_growth",
    "unemployment_rate",
    "inflation"
]

for country, df in transformed_data.items():
    
    df_std = df.with_columns([
        ((pl.col(col) - pl.col(col).mean()) / pl.col(col).std()).alias(col)
        for col in predictors
    ])
    
    standardized_data[country] = df_std

In [60]:
stan_df_stationarity_res = stationarity_check(variables=variables, data_dict=standardized_data).to_pandas()
stan_df_stationarity_res.to_csv('stan_check.csv')

C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_20400\2388354754.py:15: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(series, regression="c", nlags="auto")
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_20400\2388354754.py:15: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result = kpss(series, regression="c", nlags="auto")
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_20400\2388354754.py:15: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result = kpss(series, regression="c", nlags="auto")
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_20400\2388354754.py:15: InterpolationWarning: The test statistic is outside of th

## Multicollinearity

### Functions

In [82]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(df, predictors):
    pdf = df.select(predictors).to_pandas().dropna()
    
    vif_data = []
    
    for i in range(len(pdf.columns)):
        vif = variance_inflation_factor(pdf.values, i)
        vif_data.append({
            "variable": pdf.columns[i],
            "VIF": vif
        })
    
    return pd.DataFrame(vif_data)

def condition_index_from_polars(data: pl.DataFrame, predictors: list[str]):
    pdf = data.select(predictors).to_pandas().dropna()

    Z = (pdf - pdf.mean()) / pdf.std(ddof=0)
    Z = Z.loc[:, Z.std(ddof=0) > 0]
    X = Z.values

    eigvals = np.linalg.eigvals(X.T @ X)
    eigvals = np.real(eigvals)
    cond_idx = np.sqrt(eigvals.max() / eigvals)

    result = pl.DataFrame({
        "dimension": list(range(1, len(cond_idx) + 1)),
        "condition_index": cond_idx
    })

    return result

### VIF

In [70]:
trans_predictors = ["income_growth", "imports_growth", "exports_growth", "unemployment_rate",
    "inflation"
]
# predictors = [ "household_income", "unemployment_rate",
#     "poverty_rate", "inflation", "imports", "exports",
#     'income_growth', 'imports_growth', 'exports_growth'
# ]
vif_res = {}

for country, df in standardized_data.items():
    vif_df = calculate_vif(df, predictors=trans_predictors)
    print(f"\n--- {country} ---")
    print(vif_df)
    vif_res[country] = vif_df


--- Czech Republic ---
            variable        VIF
0      income_growth   3.697418
1     imports_growth  17.086304
2     exports_growth  15.553625
3  unemployment_rate   1.678677
4          inflation   1.252585

--- Canada ---
            variable        VIF
0      income_growth   1.665802
1     imports_growth  13.673060
2     exports_growth  13.068888
3  unemployment_rate   1.484729
4          inflation   1.936349

--- Mexico ---
            variable        VIF
0      income_growth   6.019204
1     imports_growth  11.988152
2     exports_growth   6.521261
3  unemployment_rate   1.698515
4          inflation   1.507822

--- Vietnam ---
            variable       VIF
0      income_growth  3.462431
1     imports_growth  3.876146
2     exports_growth  3.314428
3  unemployment_rate  1.046966
4          inflation  2.482538

--- Colombia ---
            variable       VIF
0      income_growth  4.521326
1     imports_growth  3.904360
2     exports_growth  3.734630
3  unemployment_rate  1

In [78]:
final_trans_df = {}

for country, data in standardized_data.items():
    df = (
        data.with_columns([
    ((pl.col("imports_growth") + pl.col("exports_growth")) / 2)
    .alias("trade_growth")
])
)

    final_trans_df[country] = df

In [ ]:
trans_predictors = ["income_growth",  "unemployment_rate",
    "inflation", 'trade_growth',
]
# predictors = [ "household_income", "unemployment_rate",
#     "poverty_rate", "inflation", "imports", "exports",
#     'income_growth', 'imports_growth', 'exports_growth'
# ]
vif_res = {}

for country, df in final_trans_df.items():
    vif_df = calculate_vif(df, predictors=trans_predictors)
    print(f"\n--- {country} ---")
    print(vif_df)
    vif_res[country] = vif_df

In [103]:
vi_result_df = pl.DataFrame(vif_res)
vi_result_df.to_pandas().to_csv(r'..\data\ecoms_results\vi_results.csv')


### Condition Index

In [101]:
results = []

for country, df in final_trans_df.items():
    ci_df = condition_index_from_polars(df, predictors)
    max_ci = ci_df["condition_index"].max()
    
    results.append({
        "country": country,
        "max_condition_index": max_ci
    })

ci_results = pl.DataFrame(results)
ci_res = ci_results.to_pandas()
ci_res.to_csv(r'..\data\ecoms_results\ci_results.csv')

## Heteroskedasticity

### Functions

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white


def breusch_pagan_test(df, predictors):
    pdf = df.select(["gdp_growth"] + predictors).to_pandas().dropna()
    
    y = pdf["gdp_growth"]
    X = pdf[predictors]
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit()
    
    bp_test = het_breuschpagan(model.resid, model.model.exog)
    
    return {
        "LM_stat": bp_test[0],
        "LM_pvalue": bp_test[1],
        "F_stat": bp_test[2],
        "F_pvalue": bp_test[3]
    }


def white_test(df, predictors):
    pdf = df.select(["gdp_growth"] + predictors).to_pandas().dropna()
    
    y = pdf["gdp_growth"]
    X = pdf[predictors]
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit()
    
    white = het_white(model.resid, model.model.exog)
    
    return {
        "LM_stat": white[0],
        "LM_pvalue": white[1],
        "F_stat": white[2],
        "F_pvalue": white[3]
    }

In [91]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white

predictors = ["income_growth", "trade_growth", "unemployment_rate", "inflation"]

models = {}
hetero_results = []

for country, data in final_trans_df.items():
    pdf = data.to_pandas().dropna()

    y = pdf["gdp_growth"]
    X = pdf[predictors]
    X = sm.add_constant(X)

    # Fit OLS model
    model = sm.OLS(y, X).fit()
    models[country] = model

    # Breusch-Pagan test
    bp_test = het_breuschpagan(model.resid, model.model.exog)

    # White test
    white_test = het_white(model.resid, model.model.exog)

    # Store results
    hetero_results.append({
        "country": country,
        "bp_lm_stat": bp_test[0],
        "bp_lm_pvalue": bp_test[1],
        "bp_f_stat": bp_test[2],
        "bp_f_pvalue": bp_test[3],
        "white_lm_stat": white_test[0],
        "white_lm_pvalue": white_test[1],
        "white_f_stat": white_test[2],
        "white_f_pvalue": white_test[3]
    })

    # Print model summary
    print(f"\n--- {country} : OLS Regression Summary ---")
    print(model.summary())

    # Print heteroskedasticity test summaries
    print(f"\n--- {country} : Breusch-Pagan Test ---")
    print({
        "LM_stat": bp_test[0],
        "LM_pvalue": bp_test[1],
        "F_stat": bp_test[2],
        "F_pvalue": bp_test[3]
    })

    print(f"\n--- {country} : White Test ---")
    print({
        "LM_stat": white_test[0],
        "LM_pvalue": white_test[1],
        "F_stat": white_test[2],
        "F_pvalue": white_test[3]
    })


--- Czech Republic : OLS Regression Summary ---
                            OLS Regression Results                            
Dep. Variable:             gdp_growth   R-squared:                       0.966
Model:                            OLS   Adj. R-squared:                  0.961
Method:                 Least Squares   F-statistic:                     171.6
Date:                Sat, 11 Apr 2026   Prob (F-statistic):           2.79e-17
Time:                        12:16:14   Log-Likelihood:                 73.853
No. Observations:                  29   AIC:                            -137.7
Df Residuals:                      24   BIC:                            -130.9
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------

In [ ]:
hetero_df = pd.DataFrame(hetero_results)
hetero_df["bp_decision"] = hetero_df["bp_lm_pvalue"].apply(
    lambda p: "Heteroskedasticity present" if p < 0.05 else "No heteroskedasticity"
)

hetero_df["white_decision"] = hetero_df["white_lm_pvalue"].apply(
    lambda p: "Heteroskedasticity present" if p < 0.05 else "No heteroskedasticity"
)

hetero_df
for col in hetero_df.columns:
    if hetero_df[col].dtype == float:
        hetero_df[col] = round(hetero_df[col], 3)

hetero_df.to_csv(r'..\data\ecoms_results\heteroskedasticity_results.csv')

In [109]:
import polars as pl
import polars.selectors as cs

final_combined_df = pl.concat([
    df.with_columns(pl.lit(country).alias("country"))
    for country, df in final_trans_df.items()
])

final_combined_df = final_combined_df.with_columns(
    cs.float().round(2)
)

final_combined_df.to_pandas().to_csv(r"..\data\ecoms_results\final_dataframe.csv")

# OLS Model Fitting

In [ ]:
predictors = ["household_income", "unemployment_rate", "poverty_rate", "inflation", "imports", "exports"]

models = {}

for country, data in transformed_country_data.items():
    pdf = data.to_pandas()

    y = pdf["gdp_growth_log"]
    X = pdf[predictors]
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()
    models[country] = model

    print(f"\n--- {country} ---")
    print(model.summary())


--- Mexico ---
                            OLS Regression Results                            
Dep. Variable:         gdp_growth_log   R-squared:                       0.922
Model:                            OLS   Adj. R-squared:                  0.900
Method:                 Least Squares   F-statistic:                     43.20
Date:                Mon, 30 Mar 2026   Prob (F-statistic):           4.52e-11
Time:                        14:57:21   Log-Likelihood:                 56.667
No. Observations:                  29   AIC:                            -99.33
Df Residuals:                      22   BIC:                            -89.76
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 

### Model Pruning - Selecting Best Predictors

In [31]:
def stepwise_model_polars(data: pl.DataFrame, target: str, predictors: list[str]):
    pdf = data.to_pandas()

    y = pdf[target]
    current_predictors = predictors.copy()

    while True:
        X = sm.add_constant(pdf[current_predictors])
        model = sm.OLS(y, X).fit()

        pvals = model.pvalues.drop("const", errors="ignore")
        max_p = pvals.max()

        if max_p > 0.05:
            worst_var = pvals.idxmax()
            current_predictors.remove(worst_var)
        else:
            break

    final_X = sm.add_constant(pdf[current_predictors])
    final_model = sm.OLS(y, final_X).fit()

    return final_model, current_predictors

In [33]:
final_models = {}
selected_predictors = {}

for country, data in transformed_country_data.items():
    final_model, kept_vars = stepwise_model_polars(
        data=data,
        target="gdp_growth_log",
        predictors=predictors
    )

    final_models[country] = final_model
    selected_predictors[country] = kept_vars

    print(f"\nFinal model for {country}")
    print("Selected predictors:", kept_vars)
    print(final_model.summary())


Final model for Mexico
Selected predictors: ['household_income', 'unemployment_rate', 'imports']
                            OLS Regression Results                            
Dep. Variable:         gdp_growth_log   R-squared:                       0.916
Model:                            OLS   Adj. R-squared:                  0.906
Method:                 Least Squares   F-statistic:                     91.20
Date:                Mon, 30 Mar 2026   Prob (F-statistic):           1.35e-13
Time:                        14:57:46   Log-Likelihood:                 55.684
No. Observations:                  29   AIC:                            -103.4
Df Residuals:                      25   BIC:                            -97.90
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
--------------------------

## Heteroskedasticity 

In [34]:
heteroskedasticity_results = []

for country, model in final_models.items():
    bp = het_breuschpagan(model.resid, model.model.exog)
    white = het_white(model.resid, model.model.exog)

    heteroskedasticity_results.append({
        "country": country,
        "bp_lm_stat": bp[0],
        "bp_p_value": bp[1],
        "white_lm_stat": white[0],
        "white_p_value": white[1]
    })

heteroskedasticity_df = pl.DataFrame(heteroskedasticity_results)
print(heteroskedasticity_df)

shape: (10, 5)
┌────────────────┬────────────┬────────────┬───────────────┬───────────────┐
│ country        ┆ bp_lm_stat ┆ bp_p_value ┆ white_lm_stat ┆ white_p_value │
│ ---            ┆ ---        ┆ ---        ┆ ---           ┆ ---           │
│ str            ┆ f64        ┆ f64        ┆ f64           ┆ f64           │
╞════════════════╪════════════╪════════════╪═══════════════╪═══════════════╡
│ Mexico         ┆ 4.687482   ┆ 0.196165   ┆ 14.910169     ┆ 0.093433      │
│ Egypt          ┆ 4.240535   ┆ 0.12       ┆ 12.834635     ┆ 0.024979      │
│ Czech Republic ┆ 2.520897   ┆ 0.112347   ┆ 2.739758      ┆ 0.254138      │
│ Canada         ┆ 4.91547    ┆ 0.178092   ┆ 16.97761      ┆ 0.030343      │
│ South Africa   ┆ 8.182954   ┆ 0.004229   ┆ 14.889373     ┆ 0.000585      │
│ Brazil         ┆ 4.860831   ┆ 0.088      ┆ 6.692774      ┆ 0.24451       │
│ Colombia       ┆ 6.573758   ┆ 0.010349   ┆ 7.431913      ┆ 0.024332      │
│ Vietnam        ┆ 1.336796   ┆ 0.512529   ┆ 6.384592      ┆ 

In [35]:
robust_models = {}

for country, model in final_models.items():
    robust_models[country] = model.get_robustcov_results(cov_type="HC1")

    print(f"\nRobust model for {country}")
    print(robust_models[country].summary())


Robust model for Mexico
                            OLS Regression Results                            
Dep. Variable:         gdp_growth_log   R-squared:                       0.916
Model:                            OLS   Adj. R-squared:                  0.906
Method:                 Least Squares   F-statistic:                     77.10
Date:                Mon, 30 Mar 2026   Prob (F-statistic):           9.07e-13
Time:                        14:58:43   Log-Likelihood:                 55.684
No. Observations:                  29   AIC:                            -103.4
Df Residuals:                      25   BIC:                            -97.90
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const        

## VIF

In [36]:
def calculate_vif_from_polars(data: pl.DataFrame, predictors: list[str]):
    pdf = data.select(predictors).to_pandas()
    X = sm.add_constant(pdf)

    vif_rows = []
    for i, col in enumerate(X.columns):
        vif_rows.append({
            "variable": col,
            "VIF": variance_inflation_factor(X.values, i)
        })

    return pl.DataFrame(vif_rows)

In [37]:
vif_results = {}

for country, data in transformed_country_data.items():
    kept_vars = selected_predictors[country]
    vif_table = calculate_vif_from_polars(data, kept_vars)
    vif_results[country] = vif_table

    print(f"\nVIF for {country}")
    print(vif_table)


VIF for Mexico
shape: (4, 2)
┌───────────────────┬──────────┐
│ variable          ┆ VIF      │
│ ---               ┆ ---      │
│ str               ┆ f64      │
╞═══════════════════╪══════════╡
│ const             ┆ 1.288588 │
│ household_income  ┆ 2.400929 │
│ unemployment_rate ┆ 1.308176 │
│ imports           ┆ 2.644231 │
└───────────────────┴──────────┘

VIF for Egypt
shape: (3, 2)
┌──────────────────┬──────────┐
│ variable         ┆ VIF      │
│ ---              ┆ ---      │
│ str              ┆ f64      │
╞══════════════════╪══════════╡
│ const            ┆ 1.26649  │
│ household_income ┆ 1.051011 │
│ poverty_rate     ┆ 1.051011 │
└──────────────────┴──────────┘

VIF for Czech Republic
shape: (2, 2)
┌──────────────────┬──────────┐
│ variable         ┆ VIF      │
│ ---              ┆ ---      │
│ str              ┆ f64      │
╞══════════════════╪══════════╡
│ const            ┆ 1.311787 │
│ household_income ┆ 1.0      │
└──────────────────┴──────────┘

VIF for Canada
shape: (4, 2)

## Condition Index

In [ ]:
def condition_index_from_polars(data: pl.DataFrame, predictors: list[str]):
    pdf = data.select(predictors).to_pandas().dropna()

    Z = (pdf - pdf.mean()) / pdf.std(ddof=0)
    Z = Z.loc[:, Z.std(ddof=0) > 0]
    X = Z.values

    eigvals = np.linalg.eigvals(X.T @ X)
    eigvals = np.real(eigvals)
    cond_idx = np.sqrt(eigvals.max() / eigvals)

    result = pl.DataFrame({
        "dimension": list(range(1, len(cond_idx) + 1)),
        "condition_index": cond_idx
    })

    return result

predictors = ["household_income", "unemployment_rate", "poverty_rate", "inflation", "imports", "exports"]
condition_index_from_polars(data=clean_df, predictors=predictors)

In [39]:
condition_index_results = []

for country, data in transformed_country_data.items():
    kept_vars = selected_predictors[country]
    ci_values = condition_index_from_polars(data, kept_vars)

    condition_index_results.append({
        "country": country,
        "max_condition_index": float(np.max(ci_values))
    })

condition_index_df = pl.DataFrame(condition_index_results)
print(condition_index_df)

shape: (10, 2)
┌────────────────┬─────────────────────┐
│ country        ┆ max_condition_index │
│ ---            ┆ ---                 │
│ str            ┆ f64                 │
╞════════════════╪═════════════════════╡
│ Mexico         ┆ 82296.706659        │
│ Egypt          ┆ 4813.551307         │
│ Czech Republic ┆ 2347.576701         │
│ Canada         ┆ 1.2011e6            │
│ South Africa   ┆ 746.985384          │
│ Brazil         ┆ 6834.209647         │
│ Colombia       ┆ 919.452516          │
│ Vietnam        ┆ 20.213963           │
│ Germany        ┆ 138444.236549       │
│ India          ┆ 91757.171733        │
└────────────────┴─────────────────────┘


In [60]:
anova_results = {}

for country, model in final_models.items():
    try:
        anova_table = anova_lm(model)
        anova_results[country] = anova_table
        print(f"\nANOVA for {country}")
        print(anova_table)
    except Exception as e:
        print(f"ANOVA not available for {country}: {e}")

ANOVA not available for Germany: 'PandasData' object has no attribute 'design_info'
ANOVA not available for Brazil: 'PandasData' object has no attribute 'design_info'
ANOVA not available for Czech Republic: 'PandasData' object has no attribute 'design_info'
ANOVA not available for Colombia: 'PandasData' object has no attribute 'design_info'
ANOVA not available for Egypt: 'PandasData' object has no attribute 'design_info'
ANOVA not available for India: 'PandasData' object has no attribute 'design_info'
ANOVA not available for Canada: 'PandasData' object has no attribute 'design_info'
ANOVA not available for Vietnam: 'PandasData' object has no attribute 'design_info'
ANOVA not available for South Africa: 'PandasData' object has no attribute 'design_info'
ANOVA not available for Mexico: 'PandasData' object has no attribute 'design_info'
